# Homework 4 — Spark basics

About 45 minutes. You explore Wavely's **users** and **artists** tables with Spark, using schemas, filters, sorting and `groupBy` from session 4.

`users.csv` columns: `user_id, signup_date, country, plan, birth_year` (birth year is optional, so sometimes empty)

In [ ]:
# Run me first: finds the course data, or downloads it when running in Google Colab.
import subprocess
from pathlib import Path

COURSE_REPO = "https://github.com/alinafazliu/data-engineering-modul-1.git"


def find_data():
    for folder in [Path.cwd(), *Path.cwd().parents, Path("/content/course")]:
        for candidate in [folder / "data", folder / "month_course" / "data"]:
            if (candidate / "reference" / "tracks.csv").exists():
                return candidate


DATA = find_data()
if DATA is None:
    subprocess.run(["git", "clone", "--depth", "1", "--quiet", COURSE_REPO, "/content/course"], check=True)
    DATA = find_data()
print("Course data:", DATA)

In [ ]:
# Start Spark. Colab already has PySpark; anywhere else we install it first.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "pyspark>=3.5,<3.6"], check=True)
os.environ["PYSPARK_PYTHON"] = sys.executable  # Spark's workers must use the same Python as the notebook

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("wavely")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")  # same behaviour on Spark 3.5 and 4.x: bad casts give null
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready")

### Exercise 1 — Load with a schema
Read `DATA / "reference" / "users.csv"` into `users` with an explicit schema.
Use `DATE` for `signup_date` and `INT` for `birth_year`.

In [ ]:
# YOUR CODE HERE

users.printSchema()
users.show(5)
print(users.count())  # expected: 800

### Exercise 2 — Free vs premium
Count users per `plan`, and users per `country`, each sorted by count (largest first).

In [ ]:
# YOUR CODE HERE

### Exercise 3 — Missing birth years
How many users have no `birth_year`? Store it in `n_missing`.

In [ ]:
# YOUR CODE HERE

print(n_missing)

### Exercise 4 — Young listeners in Austria
1. Add a column `age` = `2026 - birth_year`.
2. Keep users from `AT` aged **under 25**.
3. Show `user_id`, `plan`, `age`, youngest first, and store the number of such users in `n_young_at`.

In [ ]:
# YOUR CODE HERE

print(n_young_at)

### Exercise 5 — Sign-ups per year
`F.year(F.col("signup_date"))` extracts the year from a date. Count sign-ups per year, sorted by year.
Which year had the most sign-ups?

In [ ]:
# YOUR CODE HERE

### Exercise 6 — Artists
Load `artists.csv` (`artist_id, artist_name, country, genre`) and show the artists that are **not** from the US or GB,
sorted by name.

In [ ]:
# YOUR CODE HERE

### Exercise 7 (Stretch) — Lazy or not?
For each line below, decide **before running**: transformation or action? Then check with `explain()` on `result`.

```python
a = users.filter(F.col("plan") == "premium")
b = a.withColumn("age", 2026 - F.col("birth_year"))
c = b.count()
result = b.select("user_id", "age").orderBy("age")
result.show(3)
```

In [ ]:
# YOUR CODE HERE